# Velora — Build + Simulation
Upload the GitHub repository ZIP. This notebook builds `Velora-debug.apk` and, when the Colab runtime exposes KVM, launches a headless Android emulator and captures the main Velora Simulation Mode screens.

## 1. Upload the GitHub ZIP
On GitHub choose **Code → Download ZIP**, then run the next cell and upload that ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Using:', zip_name)

## 2. Extract and detect the Android project

In [ ]:
import os, pathlib, shutil, subprocess, zipfile, time
work = pathlib.Path('/content/velora-build')
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile('/content/' + zip_name) as z: z.extractall(work)
candidates = list(work.rglob('settings.gradle.kts')) + list(work.rglob('settings.gradle'))
if not candidates: raise RuntimeError('No Gradle Android project found in ZIP')
project = min((p.parent for p in candidates), key=lambda p: len(p.parts))
print('Project root:', project)

## 3. Install JDK + Android SDK

In [ ]:
subprocess.run('apt-get update -qq && apt-get install -y openjdk-17-jdk wget unzip > /dev/null', shell=True, check=True)
sdk = pathlib.Path('/content/android-sdk')
cmdline = sdk / 'cmdline-tools' / 'latest'
if not (cmdline / 'bin' / 'sdkmanager').exists():
    archive = pathlib.Path('/content/cmdline-tools.zip')
    subprocess.run(['wget','-q','https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip','-O',str(archive)], check=True)
    temp = pathlib.Path('/content/cmdline-temp')
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir()
    with zipfile.ZipFile(archive) as z: z.extractall(temp)
    cmdline.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(temp / 'cmdline-tools'), str(cmdline))
os.environ['ANDROID_SDK_ROOT'] = str(sdk)
os.environ['ANDROID_HOME'] = str(sdk)
os.environ['PATH'] = str(sdk / 'platform-tools') + ':' + str(sdk / 'emulator') + ':' + os.environ['PATH']
sdkmanager = str(cmdline / 'bin' / 'sdkmanager')
subprocess.run(f'yes | {sdkmanager} --licenses > /dev/null', shell=True, check=False)
subprocess.run([sdkmanager, 'platform-tools', 'platforms;android-35', 'build-tools;35.0.0'], check=True)
(project / 'local.properties').write_text('sdk.dir=' + str(sdk) + '\n')
print('Android SDK ready')

## 4. Build Velora APK

In [ ]:
gradle_home = pathlib.Path('/content/gradle-8.9')
if not gradle_home.exists():
    gradle_zip = pathlib.Path('/content/gradle.zip')
    subprocess.run(['wget','-q','https://services.gradle.org/distributions/gradle-8.9-bin.zip','-O',str(gradle_zip)], check=True)
    with zipfile.ZipFile(gradle_zip) as z: z.extractall('/content')
gradle = str(gradle_home / 'bin' / 'gradle')
env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
subprocess.run([gradle, '--no-daemon', ':app:assembleDebug'], cwd=project, env=env, check=True)
apks = list((project / 'app' / 'build' / 'outputs' / 'apk' / 'debug').glob('*.apk'))
if not apks: raise RuntimeError('Build succeeded but APK was not found')
output = pathlib.Path('/content/Velora-debug.apk')
shutil.copy2(apks[0], output)
print('APK:', output, 'size:', round(output.stat().st_size/1024/1024, 2), 'MB')

## 5. Optional local Colab emulator + screenshots
Leave `RUN_LOCAL_EMULATOR = True`. The notebook checks KVM first. If this Colab runtime does not expose KVM, it safely skips the emulator and keeps the APK.

In [ ]:
RUN_LOCAL_EMULATOR = True
preview_zip = None
has_kvm = pathlib.Path('/dev/kvm').exists() and os.access('/dev/kvm', os.R_OK | os.W_OK)
print('KVM available:', has_kvm)

if RUN_LOCAL_EMULATOR and has_kvm:
    system_image = 'system-images;android-35;google_apis;x86_64'
    subprocess.run([sdkmanager, 'emulator', system_image], check=True)
    avdmanager = str(cmdline / 'bin' / 'avdmanager')
    android_home = pathlib.Path('/content/.android')
    android_home.mkdir(exist_ok=True)
    env['ANDROID_AVD_HOME'] = str(android_home / 'avd')
    pathlib.Path(env['ANDROID_AVD_HOME']).mkdir(parents=True, exist_ok=True)
    subprocess.run(
        f'echo no | "{avdmanager}" create avd -n velora_api35 -k "{system_image}" --device "pixel_6" --force',
        shell=True, env=env, check=True
    )
    emulator = str(sdk / 'emulator' / 'emulator')
    log = open('/content/velora-emulator.log', 'w')
    emulator_proc = subprocess.Popen([
        emulator, '-avd', 'velora_api35', '-no-window', '-no-audio',
        '-no-boot-anim', '-no-snapshot', '-wipe-data', '-gpu', 'swiftshader_indirect'
    ], stdout=log, stderr=subprocess.STDOUT, env=env)
    adb = str(sdk / 'platform-tools' / 'adb')
    subprocess.run([adb, 'wait-for-device'], check=True)
    booted = False
    for _ in range(180):
        state = subprocess.run([adb, 'shell', 'getprop', 'sys.boot_completed'], capture_output=True, text=True).stdout.strip()
        if state == '1':
            booted = True
            break
        time.sleep(2)
    if not booted:
        raise RuntimeError('Emulator did not finish booting. See /content/velora-emulator.log')
    subprocess.run([adb, 'install', '-r', str(output)], check=True)
    preview_dir = pathlib.Path('/content/velora-previews')
    if preview_dir.exists(): shutil.rmtree(preview_dir)
    preview_dir.mkdir()
    for screen in ['home', 'drawer', 'control', 'settings', 'widgets']:
        subprocess.run([adb, 'shell', 'am', 'force-stop', 'tech.wonderer.velora'], check=False)
        subprocess.run([
            adb, 'shell', 'am', 'start', '-W',
            '-n', 'tech.wonderer.velora/.SimulationActivity',
            '--es', 'screen', screen
        ], check=True, stdout=subprocess.DEVNULL)
        time.sleep(2)
        with open(preview_dir / f'velora-{screen}.png', 'wb') as image_file:
            subprocess.run([adb, 'exec-out', 'screencap', '-p'], stdout=image_file, check=True)
    preview_zip = pathlib.Path(shutil.make_archive('/content/Velora-previews', 'zip', preview_dir))
    print('Preview screenshots:', preview_zip)
else:
    print('Local emulator skipped. The APK still contains the Velora Preview simulation entry for cloud/browser emulators.')

## 6. Download outputs
The APK always downloads. If the local emulator ran, the screenshot ZIP downloads too.

In [ ]:
from google.colab import files
files.download('/content/Velora-debug.apk')
if preview_zip is not None and pathlib.Path(preview_zip).exists():
    files.download(str(preview_zip))